# CPU tracking quickstart

Create a workspace and save its key first. Run cells in order. Credentials use hidden prompts.


In [ ]:
%pip install -q "crashlens[jupyter,gpu] @ git+https://github.com/kavinsaravan/ML-Failure-Doctor.git#subdirectory=crashlens-sdk"

In [ ]:
import os, getpass, requests
from crashlens import WorkloadTracker
api_url = (os.getenv("CRASHLENS_URL") or input("Backend HTTPS URL: ")).strip().rstrip("/")
api_key = os.getenv("CRASHLENS_API_KEY") or getpass.getpass("Your saved CrashLens key: ")
fireworks_key = os.getenv("FIREWORKS_API_KEY") or getpass.getpass("Your Fireworks key (blank for rules): ")
fireworks_model = (os.getenv("FIREWORKS_MODEL") or input("Tool-capable Fireworks model: ")) if fireworks_key else None
tracker = WorkloadTracker(api_url, api_key=api_key, fireworks_api_key=fireworks_key or None, fireworks_model=fireworks_model)
response = requests.get(api_url + "/session", headers=tracker.headers, timeout=15)
response.raise_for_status()
print("Connected to your private workspace")

In [ ]:
with tracker.track("CPU calculation") as workload_id:
    print("CPU result:", sum(i * i for i in range(100000)))
print("Workload ID:", workload_id)

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
try:
    with tracker.track("Missing checkpoint") as failed_id:
        with TemporaryDirectory() as folder:
            Path(folder, "nonexistent-checkpoint.pt").read_bytes()
except FileNotFoundError:
    print("Expected missing-file failure recorded; workload", failed_id)
report = tracker.diagnose(failed_id)  # Uses your credits only if Fireworks credentials are set.
print(report["source"], report["root_cause"])

In [ ]:
response = requests.get(api_url + "/workloads", headers=tracker.headers, timeout=15)
response.raise_for_status()
for workload in response.json():
    print(workload["id"], workload["name"], workload["status"])
print("Open your frontend dashboard using the same CrashLens key.")